In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os

folder_path = '/content/drive/MyDrive/t5_squad_project_16x2'
print(os.listdir(folder_path))


['train-v1.1.json']


In [6]:
import os
import json
import re
import torch
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from torch.utils.data import Dataset
from transformers import TrainerCallback
import zipfile

# تعطيل WandB
os.environ["WANDB_DISABLED"] = "true"

# مجلد الحفظ باسم جديد
base_dir = "/content/drive/MyDrive/t5_squad_project_16x2"
os.makedirs(base_dir, exist_ok=True)

# كولباك لتسجيل الخسارة
class LossLoggerCallback(TrainerCallback):
    def __init__(self, log_file_path):
        self.log_file_path = log_file_path

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % 1000 == 0 or state.global_step == 1:
            train_loss = logs.get("loss")
            eval_loss = logs.get("eval_loss")
            step = state.global_step
            print(f"Step {step} - Train Loss: {train_loss} - Eval Loss: {eval_loss}")
            with open(self.log_file_path, "a") as f:
                f.write(f"Step {step} - Train Loss: {train_loss} - Eval Loss: {eval_loss}\n")

# تنظيف النصوص
def clean_text(text):
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

# إزالة التكرارات
def remove_duplicate_questions(data):
    seen = set()
    unique_data = []
    for item in data:
        if item['cleaned_question'] not in seen:
            seen.add(item['cleaned_question'])
            unique_data.append(item)
    return unique_data

# تحميل بيانات SQuAD
def read_and_process_squad_data(file_path, limit=None):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print("حدث خطأ:", e)
        return []

    processed_data = []
    for article in data['data']:
        for paragraph in article['paragraphs']:
            for qa in paragraph['qas']:
                question = clean_text(qa['question'])
                answers = [clean_text(a['text']) for a in qa['answers']]
                processed_data.append({
                    'context': paragraph['context'],
                    'cleaned_question': question,
                    'cleaned_answers': answers
                })
    processed_data = remove_duplicate_questions(processed_data)
    if limit:
        processed_data = processed_data[:limit]
    return train_test_split(processed_data, test_size=0.2, random_state=42)


# Dataset مخصص لتوليد الأسئلة
class SquadDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        context = item['context']
        question = item['cleaned_question']  # السؤال هو الهدف

        inputs = self.tokenizer(
            f"context: {context}",
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            question,
            max_length=32,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': targets['input_ids'].squeeze()
        }

# تحميل النموذج والـ tokenizer
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# الجهاز (GPU أو CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# تحميل بيانات SQuAD
file_path = "/content/drive/MyDrive/t5_squad_project_16x2/train-v1.1.json"

train_data, test_data = read_and_process_squad_data(file_path)

# تجهيز البيانات
train_dataset = SquadDataset(train_data, tokenizer)
test_dataset = SquadDataset(test_data, tokenizer)

# إعدادات التدريب
training_args = TrainingArguments(
    output_dir=f"{base_dir}/output_results_16x2",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,  # لتجميع gradients
    num_train_epochs=5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=f"{base_dir}/logs_16x2",
    logging_steps=1000,
    save_strategy="steps",
    eval_strategy="steps",
    save_steps=1000,
    eval_steps=1000,
    save_total_limit=1,
    load_best_model_at_end=True,
)

# سجل الخسارة
log_file_path = f"{base_dir}/training_log_16x2.txt"

# المدرب
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[LossLoggerCallback(log_file_path)]
)

# بدء التدريب
print("بدء التدريب...")
trainer.train()

# حفظ النموذج و tokenizer
new_model_path = f"{base_dir}/t5_squad_model_final_16x2"
trainer.save_model(new_model_path)
tokenizer.save_pretrained(new_model_path)

print(f"تم حفظ النموذج في: {new_model_path}")
print(f"سجل الخسارة محفوظ في: {log_file_path}")

# ضغط النتائج
zip_path = f"{base_dir}/t5_squad_backup_16x2.zip".
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for foldername, subfolders, filenames in os.walk(new_model_path):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            arcname = os.path.relpath(file_path, base_dir)
            zipf.write(file_path, arcname)
    zipf.write(log_file_path, os.path.relpath(log_file_path, base_dir))

print(f"تم ضغط الملفات في: {zip_path}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


بدء التدريب...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
1000,1.742500,1.003341
2000,1.099100,0.961226
3000,1.060600,0.939211
4000,1.035200,0.925802
5000,1.015100,0.917413
6000,1.004500,0.910443
7000,1.004300,0.904912
8000,0.987100,0.902406
9000,0.983400,0.899377


Step 1000 - Train Loss: 1.7425 - Eval Loss: None
Step 1000 - Train Loss: None - Eval Loss: 1.003340721130371
Step 2000 - Train Loss: 1.0991 - Eval Loss: None
Step 2000 - Train Loss: None - Eval Loss: 0.9612258076667786
Step 3000 - Train Loss: 1.0606 - Eval Loss: None
Step 3000 - Train Loss: None - Eval Loss: 0.939211368560791
Step 4000 - Train Loss: 1.0352 - Eval Loss: None
Step 4000 - Train Loss: None - Eval Loss: 0.9258019924163818
Step 5000 - Train Loss: 1.0151 - Eval Loss: None
Step 5000 - Train Loss: None - Eval Loss: 0.9174128174781799
Step 6000 - Train Loss: 1.0045 - Eval Loss: None
Step 6000 - Train Loss: None - Eval Loss: 0.9104427695274353
Step 7000 - Train Loss: 1.0043 - Eval Loss: None
Step 7000 - Train Loss: None - Eval Loss: 0.9049120545387268
Step 8000 - Train Loss: 0.9871 - Eval Loss: None
Step 8000 - Train Loss: None - Eval Loss: 0.9024057984352112
Step 9000 - Train Loss: 0.9834 - Eval Loss: None
Step 9000 - Train Loss: None - Eval Loss: 0.899377167224884


Step,Training Loss,Validation Loss
1000,1.742500,1.003341
2000,1.099100,0.961226
3000,1.060600,0.939211
4000,1.035200,0.925802
5000,1.015100,0.917413
6000,1.004500,0.910443
7000,1.004300,0.904912
8000,0.987100,0.902406
9000,0.983400,0.899377
10000,0.978500,0.897458


Step 10000 - Train Loss: 0.9785 - Eval Loss: None
Step 10000 - Train Loss: None - Eval Loss: 0.8974575996398926


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


تم حفظ النموذج في: /content/drive/MyDrive/t5_squad_project_16x2/t5_squad_model_final_16x2
سجل الخسارة محفوظ في: /content/drive/MyDrive/t5_squad_project_16x2/training_log_16x2.txt
تم ضغط الملفات في: /content/drive/MyDrive/t5_squad_project_16x2/t5_squad_backup_16x2.zip


In [11]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# تحميل النموذج و tokenizer من مسار محفوظ
model_path = "/content/drive/MyDrive/t5_squad_project_16x2/t5_squad_model_final_16x2"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# دالة لتوليد الأسئلة لدفعة من السياقات
def generate_questions_batch(contexts, batch_size=8, max_length=256):
    all_questions = []
    for i in range(0, len(contexts), batch_size):
        batch = contexts[i:i+batch_size]
        inputs = tokenizer([f"context: {c}" for c in batch],
                           return_tensors="pt",
                           padding=True,
                           truncation=True,
                           max_length=max_length).to(device)
        outputs = model.generate(
            **inputs,
            max_length=32,
            num_beams=5,
            early_stopping=True
        )
        questions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        all_questions.extend(questions)
    return all_questions

# مثال: قائمة سياقات (Contexts)
contexts = [
    "The Apollo 11 mission was the first to land humans on the Moon.",
    "The Great Wall of China is one of the wonders of the world.",
    "Python is a popular programming language for AI and machine learning.",
    # ضيف المزيد من الفقرات حسب بياناتك...
]

# توليد الأسئلة دفعة دفعة
questions = generate_questions_batch(contexts, batch_size=2)
for i, q in enumerate(questions):
    print(f"Context {i+1}: {contexts[i]}")
    print(f"Generated Question: {q}\n")


Context 1: The Apollo 11 mission was the first to land humans on the Moon.
Generated Question: what mission was the first to land humans on the moon

Context 2: The Great Wall of China is one of the wonders of the world.
Generated Question: what is one of the wonders of the world

Context 3: Python is a popular programming language for AI and machine learning.
Generated Question: what is python a popular programming language for



In [16]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# المسار اللي حفظت فيه النموذج
model_path = "/content/drive/MyDrive/t5_squad_project_16x2/t5_squad_model_final_16x2"

# تحميل النموذج  tokenizer
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# مثال على سياق (context) للاختبار
context = "The Apollo 11 mission was the first to land humans on the Moon."

# تجهيز الإدخال للنموذج
input_text = f"context: {context}"
inputs = tokenizer(input_text, return_tensors="pt", max_length=256, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

# توليد السؤال
outputs = model.generate(
    **inputs,
    max_length=32,
    num_beams=5,  # لتحسين جودة التوليد
    early_stopping=True
)

# تحويل التوكنات الناتجة إلى نص
generated_question = tokenizer.decode(outputs[0], skip_special_tokens=True)

# التأكد من وجود علامة استفهام في نهاية السؤال
if not generated_question.strip().endswith('?'):
    generated_question = generated_question.strip() + '?'

print("السؤال المُولد:", generated_question)


السؤال المُولد: what mission was the first to land humans on the moon?


In [17]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# تحميل النموذج و tokenizer من مسار محفوظ
model_path = "/content/drive/MyDrive/t5_squad_project_16x2/t5_squad_model_final_16x2"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# دالة لتوليد الأسئلة لدفعة من السياقات مع إضافة علامة استفهام إذا لم تكن موجودة
def generate_questions_batch(contexts, batch_size=8, max_length=256):
    all_questions = []
    for i in range(0, len(contexts), batch_size):
        batch = contexts[i:i+batch_size]
        inputs = tokenizer([f"context: {c}" for c in batch],
                           return_tensors="pt",
                           padding=True,
                           truncation=True,
                           max_length=max_length).to(device)
        outputs = model.generate(
            **inputs,
            max_length=32,
            num_beams=5,
            early_stopping=True
        )
        questions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        # إضافة علامة استفهام إن لم تكن موجودة
        questions = [q if q.strip().endswith('?') else q.strip() + '?' for q in questions]
        all_questions.extend(questions)
    return all_questions

# مثال: قائمة سياقات (Contexts)
contexts = [
    "The Apollo 11 mission was the first to land humans on the Moon.",
    "The Great Wall of China is one of the wonders of the world.",
    "Python is a popular programming language for AI and machine learning.",
    # ضيف المزيد من الفقرات حسب بياناتك...
]

# توليد الأسئلة دفعة دفعة
questions = generate_questions_batch(contexts, batch_size=2)
for i, q in enumerate(questions):
    print(f"Context {i+1}: {contexts[i]}")
    print(f"Generated Question: {q}\n")


Context 1: The Apollo 11 mission was the first to land humans on the Moon.
Generated Question: what mission was the first to land humans on the moon?

Context 2: The Great Wall of China is one of the wonders of the world.
Generated Question: what is one of the wonders of the world?

Context 3: Python is a popular programming language for AI and machine learning.
Generated Question: what is python a popular programming language for?

